In [ ]:
Tools are simply functions or external services that an LLM can use to do real-world tasks. On its own,
an LLM can only generate text, but with tools it can fetch live data, perform accurate calculations, access databases, or
execute actions. in short, tools extend an LLM from just thinking to actually doing things
* we use an LLM as the brain, tools to perform tasks, and our code to define the logic together, this creates an agent.
* LLM decides what to do, tools do the tasks, and our code controls how everything works together.
* Agent = Brain(LLM)+ Tools + Control Logic(our code)

In [ ]:
Tools Category ---> (builds-in Tools) AND (custom Tools)

In [2]:
# Build in tools (the tools which provided by frameworks, so you don't have to create them form scratch)
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults

# Correct way to initialize the TavilySearchResults tool
# The parameter should be 'tavily_api_key', not 'TAVILY_API_KEY'
search_tool = TavilySearchResults(
    max_results=2,  # Corrected from 'max_result' to 'max_results'
    tavily_api_key="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"
)

llm = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

# 2. Define your prompt
prompt = ChatPromptTemplate.from_template(""" you are a helpful assistant
 summarize the following news into clear bullet poits 
 {news}
""")

parser = StrOutputParser()

# 3. Create and run the chain
chain = prompt | llm | parser

news_result = search_tool.invoke('Latest AI news of 2026')  # Changed from .run() to .invoke()

# Pass the actual news_result variable, not the string "news_result"
response = chain.invoke({"news": news_result})
print(response)

Here are the clear bullet points summarizing the news about AI and tech trends in 2026:

**General Trends:**
* AI capabilities will pave new ways to do business in the enterprise.
* Trust and security will become key priorities as many enterprises sharpen their focus on AI sovereignty.
* Open-source reasoning models and agents will keep pushing boundaries to conquer enterprise AI.

**Open-Source AI:**
* Global model diversification, led by Chinese multilingual and reasoning-tuned releases, will define open-source AI in 2026.
* Interoperability will become a competitive axis, with frameworks and runtimes aligning around shared standards.
* Hardened governance, with security-audited releases and transparent data pipelines, will be a key focus.
* Smaller, domain-specific models will achieve impressive results and become more prevalent.

**Quantum Computing:**
* Quantum computing will start tackling problems that classical computers can't, with a looming breakthrough called quantum advanta

In [6]:
# custom Tools (the tolls that you create yourself based on your needs.)
from langchain.tools import tool

@tool
def get_greeting(name:str)-> str:
    """Generate a greeting message for a user"""
    return f"hello {name}, welcome to the Ai world"

result=get_greeting.invoke({"name":"rishabh"})
print(result)
     
print(get_greeting.name)
print(get_greeting.description)
print(get_greeting.args)

hello rishabh, welcome to the Ai world
get_greeting
Generate a greeting message for a user
{'name': {'title': 'Name', 'type': 'string'}}


In [1]:
# Build in tools (the tools which provided by frameworks, so you don't have to create them form scratch)
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
from rich import print
from langchain_core.messages import HumanMessage

@tool
def get_text_length(text:str)-> int:
    """Returns the number of character in a given text"""
    return len(text)

tools ={
    "get_text_length": get_text_length
}

llm = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

llm_with_tool = llm.bind_tools([get_text_length])

message =[]

query = HumanMessage("Returns the number of character in given text: 'hello'")
message.append(query)

result = llm_with_tool.invoke(message)
message.append(result)

if result.tool_calls:
    print(result.tool_calls[0])
    tool_name= result.tool_calls[0]['name']
    tool_message=tools[tool_name].invoke(result.tool_calls[0])
    message.append(tool_message)
    print(message)


{'name': 'get_text_length', 'args': {'text': 'hello'}, 'id': 'se220azvk', 'type': 'tool_call'}

[
    HumanMessage(
        content="Returns the number of character in given text: 'hello'",
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(
        content='',
        additional_kwargs={
            'tool_calls': [
                {
                    'id': 'se220azvk',
                    'function': {'arguments': '{"text":"hello"}', 'name': 'get_text_length'},
                    'type': 'function'
                }
            ]
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 15,
                'prompt_tokens': 232,
                'total_tokens': 247,
                'completion_time': 0.03698305,
                'completion_tokens_details': None,
                'prompt_time': 0.026606834,
                'prompt_tokens_details': None,
                'queue_time': 0.079562074,
                'total_time': 0.063589884
            },
            'model_name': 'llama-3.3-70b-versatile',
            'system_fingerprint': 'fp_dae98b5ecb',
            'service_tier': 'on_demand',
            'finish_reason': 'tool_calls',
            'logprobs': None,
            'model_provider': 'groq'
        },
        id='lc_run--019d597f-73a6-7e32-822f-dd228cb93177-0',
        tool_calls=[
            {'name': 'get_text_length', 'args': {'text': 'hello'}, 'id': 'se220azvk', 'type': 'tool_call'}
        ],
        invalid_tool_calls=[],
        usage_metadata={'input_tokens': 232, 'output_tokens': 15, 'total_tokens': 247}
    ),
    ToolMessage(content='5', name='get_text_length', tool_call_id='se220azvk')
]

In [14]:
note
********tool binding
tool binding means connecting our tools to the LLM so it can use the when needed
we give our tools to the LLM, and now the LLM knows:
    .what tools are available
    .what they do
    .when to use them

********tool calling
in tool binding,we gave tool to the llm
in tool calling, the LLM chooses to use them.

theLLM does not execute the tool it only suggest the tool call.

*******tool execution
tool execution is when we actually run the tool after the LLM decides to use it.
LLM selects the tool---> we execute it----> get the result ---> send it back to the LLM.

SyntaxError: invalid syntax (3487933689.py, line 2)

In [ ]:
import requests
from langchain.tools import tool
from langchain_core.messages import ToolMessage
from tavily import TavilyClient
from rich import print
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_groq import ChatGroq

# =========================
# 🌦️ Weather Tool
# =========================

@tool
def get_weather(city: str) -> str:
    """Get current weather of a city"""
    api_key = "83b764eec15b3f3aed3ab569d7bffe63"
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city},IN&appid={api_key}&units=metric"
    
    response = requests.get(url)
    data = response.json()
    
    if str(data.get("cod")) != "200":
        return f"Error: {data.get('message', 'Could not fetch weather')}"
    
    temp = data["main"]["temp"]
    desc = data["weather"][0]["description"]
    
    return f"Weather in {city}: {desc}, {temp}°C"


# =========================
# 📰 News Tool (Tavily)
# =========================

# Initialize the Tavily client
tavily_client = TavilyClient(api_key="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")

@tool
def get_news(city: str) -> str:
    """Get latest news about a city"""
    response = tavily_client.search(
        query=f"latest news in {city}",
        search_depth="basic",
        max_results=3
    )
    
    results = response.get("results", [])
    
    if not results:
        return f"No news found for {city}"
    
    news_list = []
    for r in results:
        title = r.get("title", "No title")
        url = r.get("url", "")
        snippet = r.get("content", "")
        news_list.append(f"- {title}\n  🔗 {url}\n  📝 {snippet[:100]}...")
    
    return f"Latest news in {city}:\n\n" + "\n\n".join(news_list)


# =========================
# 🧠 LLM Setup
# =========================

llm = ChatGroq(
    groq_api_key="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",
    model_name="llama-3.3-70b-versatile",
    temperature=0.5,
)


# =========================
# ✅ Human Approval Middleware
# =========================

@wrap_tool_call# dynamically choose tool
def human_approval(request, handler):
    """Ask for human approval before every tool call."""
    tool_name = request.tool_call["name"]
    confirm = input(f"Agent wants to call '{tool_name}'. Approve? (yes/no): ")

    if confirm.lower() != "yes":
        return ToolMessage(
            content="Tool call denied by user.",
            tool_call_id=request.tool_call["id"]
        )

    return handler(request)


# =========================
# 🤖 Agent Creation (new API)
# =========================

agent = create_agent(
    llm,
    tools=[get_weather, get_news], 
    system_prompt="You are a helpful city assistant.",
    middleware=[human_approval]
)

# =========================
# 🎮 Main Loop
# =========================

print("City Agent | type 'exit' to quit")

while True:
    user_input = input("You : ")
    if user_input.lower() == "exit":
        break
    
    # Invoke the agent with the user message
    result = agent.invoke({
        "messages": [{"role": "user", "content": user_input}]
    })
    
    # Print the last assistant message
    print("Bot :", result['messages'][-1].content)

City Agent | type 'exit' to quit

You :  can you tell me weather of ahmedabad and current news


In [ ]:
create_agent  is a high-level abstraction provided by langchain that automates everything we manually implemented in our 
agent. instead of writing the loop. handling tool calls, managing messages, and deciding when to stop , we simply define the model and the tools,and
the framework takes care of the entire execution process intentionally. the core logic remains the same the LLM still decide which tool to use.
tools still fetch external data, and results are passed back for final response generation but all of this happens behind the scenes. the main difference
is that our manual approach gives full control and visibility

In [ ]:
import requests
from langchain.tools import tool
from langchain_core.messages import ToolMessage
from tavily import TavilyClient
from rich import print
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_groq import ChatGroq

@tool
def getweather(city:str)->str:
    """Get current weather of a city"""
    api_key = "83b764eec15b3f3aed3ab569d7bffe63"
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city},IN&appid={api_key}&units=metric"
    response = requests.get(url)
    data = response.json()
    
    if str(data.get("cod")) != "200":
        return f"Error: {data.get('message', 'Could not fetch weather')}"
    
    temp = data["main"]["temp"]
    desc = data["weather"][0]["description"]
    
    return f"Weather in {city}: {desc}, {temp}°C"
    response = request.get(url)
    data= response.json()

# Initialize the Tavily client
tavily_client = TavilyClient(api_key="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")

@tool
def get_news(city: str) -> str:
    """Get latest news about a city"""
    response = tavily_client.search(
        query=f"latest news in {city}",
        search_depth="basic",
        max_results=3
    )
    
    results = response.get("results", [])
    
    if not results:
        return f"No news found for {city}"
    
    news_list = []
    for r in results:
        title = r.get("title", "No title")
        url = r.get("url", "")
        snippet = r.get("content", "")
        news_list.append(f"- {title}\n  🔗 {url}\n  📝 {snippet[:100]}...")
    
    return f"Latest news in {city}:\n\n" + "\n\n".join(news_list)

llm = ChatGroq(
    groq_api_key="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",
    model_name="llama-3.3-70b-versatile",
    temperature=0.5,
)


# =========================
# ✅ Human Approval Middleware
# =========================

@wrap_tool_call# dynamically choose tool
def human_approval(request, handler):
    """Ask for human approval before every tool call."""
    tool_name = request.tool_call["name"]
    confirm = input(f"Agent wants to call '{tool_name}'. Approve? (yes/no): ")

    if confirm.lower() != "yes":
        return ToolMessage(
            content="Tool call denied by user.",
            tool_call_id=request.tool_call["id"]
        )

    return handler(request) 

# =========================
# 🤖 Agent Creation (new API)
# =========================

agent = create_agent(
    llm,
    tools=[get_weather, get_news], 
    system_prompt="You are a helpful city assistant.",
    middleware=[human_approval]
)


# =========================
# 🎮 Main Loop
# ====================

print("City Agent | type 'exit' to quit")

while True:
    user_input = input("You : ")
    if user_input.lower() == "exit":
        break
    
    # Invoke the agent with the user message
    result = agent.invoke({
        "messages": [{"role": "user", "content": user_input}]
    })
    
    # Print the last assistant message
    print("Bot :", result['messages'][-1].content)


